**[Course Home](../README.md) | [Python Lab](../python/02_poisson_processes_and_memoryless_arrivals.ipynb) | [Prev: Ch 01: Conservation of Flow](01_little_law_and_conservation_of_flow.ipynb) | [Next: Ch 03: Birth-Death Chains](03_birth_death_chains_and_mm1_mminf.ipynb)**

*Part I: Foundations of Flow & Memoryless Systems — Chapter 02 of 11*

---

# Chapter 2: Random Arrivals — Poisson Processes, Exponential Inter-Arrivals, Merging & Thinning, and PASTA
### *The Law of Rare Events, The Memoryless Invariant, Stream Superposition, and Why Poisson Arrivals See Time Averages*

---

## 1. The Core Engineering Problem: Why Assume Poisson Arrivals?

In the study of queueing systems, network architectures, and distributed fleets, the **Poisson arrival process** (denoted by $M$ for "Markovian" in Kendall's notation) is the standard baseline model for traffic generation. 

Engineers often ask:

> *Why do we model developer pull requests, API requests, and webhook triggers as Poisson processes? Is it merely mathematical convenience, or does it reflect physical reality?*

The justification rests on the **Palm-Khinchin Theorem** (the queueing equivalent of the Central Limit Theorem), also known as the **Law of Rare Events**:

```
Developer 1  ---|---|-----------|-------|--------|---->   p << 1
Developer 2  -------|-------|---------------|--------->   p << 1
Developer 3  ---|---------------|---|---|------------->   p << 1
...
Developer N  -----------|---|---------------|---|----->   p << 1
               \    \   \   \   \   \   \   \   \
                v    v   v   v   v   v   v   v   v
Aggregated   ---||--||--|---|---|---|---|---|---|----->   POISSON STREAM
Traffic Stream                                            Rate Lambda = N * p
```

When an aggregate arrival stream is formed by the superposition of a very large number $N$ of independent sources, each of which has a very small probability $p$ of generating a request in any small time slice $\Delta t$, the aggregate stream converges weakly to a **Poisson process** with rate $\lambda = N p$, regardless of the underlying probability distributions of the individual sources!

In an engineering organization:
* Hundreds or thousands of software engineers are independently writing, testing, and pushing code.
* Thousands of automated bots, dependency updaters, and scheduled integration suites fire throughout the day.
* No single actor dominates the global arrival schedule.

Whenever traffic is the aggregate result of many independent, uncoordinated actors, the arrival stream is naturally Poisson. Conversely, whenever traffic is driven by a single centralized scheduler (such as an hourly cron job triggering 500 builds simultaneously), the Poisson assumption fails catastrophically.

---

## 2. Formal Mathematical Definition of the Poisson Counting Process

A stochastic counting process $\{N(t), t \ge 0\}$ represents the total cumulative number of arrivals that have occurred up to time $t$. 

A counting process is a **homogeneous Poisson process** with rate $\lambda > 0$ if and only if it satisfies four rigorous axioms:

1. **Initial Condition**: $N(0) = 0$.
2. **Independent Increments**: The number of arrivals occurring in non-overlapping time intervals are mutually independent. For any sequence of times $0 \le t_1 < t_2 < t_3 < t_4$:
   $$
   N(t_2) - N(t_1) \quad \text{and} \quad N(t_4) - N(t_3) \quad \text{are independent.}
   $$
   Past arrivals give zero information about future arrivals.
3. **Stationary Increments**: The probability distribution of the number of arrivals in any interval $[s, s + t]$ depends solely on the duration of the interval $t$, not on the start time $s$:
   $$
   N(s + t) - N(s) \sim N(t) - N(0)
   $$
4. **Infinitesimal Linearity & Non-Simultaneity**: In an infinitesimal interval of duration $\Delta t \to 0$:
   $$
   \mathbb{P}(N(t + \Delta t) - N(t) = 1) = \lambda \Delta t + o(\Delta t)
   $$
   $$
   \mathbb{P}(N(t + \Delta t) - N(t) \ge 2) = o(\Delta t)
   $$
   $$
   \mathbb{P}(N(t + \Delta t) - N(t) = 0) = 1 - \lambda \Delta t + o(\Delta t)
   $$
   Arrivals occur singly; two or more arrivals cannot occur at the exact same physical instant.

### The Poisson Probability Mass Function (PMF)
From these four axioms, solving the differential equations of flow yields the probability of observing exactly $k$ arrivals in any interval of length $t$:

$$
\mathbb{P}(N(t) = k) = \frac{(\lambda t)^k e^{-\lambda t}}{k!}, \quad k \in \{0, 1, 2, \dots\}
$$

The expected count and variance over interval $t$ are equal:

$$
\mathbb{E}[N(t)] = \lambda t
$$

$$
\text{Var}(N(t)) = \lambda t
$$

The ratio of variance to mean (the **Fano Factor**) is identically 1.0:

$$
F = \frac{\text{Var}(N(t))}{\mathbb{E}[N(t)]} = 1.0
$$

* If $F > 1$: The arrival process is **overdispersed** (bursty, clustered).
* If $F < 1$: The arrival process is **underdispersed** (regular, smoothed, paced).

---

## 3. The Exponential Distribution & The Memoryless Property

What is the probability distribution of the time between consecutive arrivals in a Poisson process?

Let $X$ denote the inter-arrival duration between two successive events. The event $\{X > t\}$ occurs if and only if **zero arrivals** occur in the interval of duration $t$:

$$
\mathbb{P}(X > t) = \mathbb{P}(N(t) = 0) = \frac{(\lambda t)^0 e^{-\lambda t}}{0!} = e^{-\lambda t}
$$

The Cumulative Distribution Function (CDF) is therefore:

$$
F(t) = \mathbb{P}(X \le t) = 1 - e^{-\lambda t}, \quad t \ge 0
$$

Differentiating $F(t)$ yields the Probability Density Function (PDF) of the **Exponential Distribution**:

$$
f(t) = \frac{d}{dt}F(t) = \lambda e^{-\lambda t}, \quad t \ge 0
$$

```
Probability Density f(t)
    ^
    |  *
    |   \
    |    \
    |     \
    |      \
    |       \
    |        \---_
    |             \______
    +------------------------------------> Inter-Arrival Time t
    0                     1/lambda
```

The mean and variance of the inter-arrival time are:

$$
\mathbb{E}[X] = \int_0^\infty t \lambda e^{-\lambda t} \, dt = \frac{1}{\lambda}
$$

$$
\text{Var}(X) = \frac{1}{\lambda^2} \implies \sigma_X = \frac{1}{\lambda}
$$

The coefficient of variation is:

$$
C_v = \frac{\sigma_X}{\mathbb{E}[X]} = \frac{1/\lambda}{1/\lambda} = 1.0
$$

### The Memoryless Invariant (Markov Property)
The exponential distribution is the **only continuous probability distribution that possesses the memoryless property**:

$$
\mathbb{P}(X > t + s \mid X > t) = \mathbb{P}(X > s)
$$

### Mathematical Proof:
Using Bayes' rule and the definition of conditional probability:

$$
\mathbb{P}(X > t + s \mid X > t) = \frac{\mathbb{P}(X > t + s \text{ and } X > t)}{\mathbb{P}(X > t)} = \frac{\mathbb{P}(X > t + s)}{\mathbb{P}(X > t)}
$$

Substitute the exponential survival function $\mathbb{P}(X > u) = e^{-\lambda u}$:

$$
\frac{e^{-\lambda(t + s)}}{e^{-\lambda t}} = \frac{e^{-\lambda t} \cdot e^{-\lambda s}}{e^{-\lambda t}} = e^{-\lambda s} = \mathbb{P}(X > s)
$$

### Physical Intuition for Engineers:
Suppose you are monitoring a CI runner queue where PRs arrive as a Poisson process with rate $\lambda = 0.1$ PRs/min (mean inter-arrival time $\mathbb{E}[X] = 10$ minutes).
* You check the queue. The last PR arrived 25 minutes ago.
* How much longer should you expect to wait for the next PR?
* An intuitive human guess might be: "It's long overdue! The next PR must arrive in the next 1 or 2 minutes."
* **The mathematical truth**: Because the process is memoryless, the remaining wait time until the next PR is still an exponential random variable with mean **10 minutes**. The fact that you have already waited 25 minutes gives zero credit toward the future.

---

## 4. Algebraic Operations: Superposition (Merging) & Splitting (Thinning)

Two fundamental algebraic operations govern Poisson streams in distributed networks:

### 1. Superposition (Merging Independent Streams)
Suppose $K$ independent services generate requests as independent Poisson processes with rates $\lambda_1, \lambda_2, \dots, \lambda_K$.

When these streams are combined into a single unified queue:
* The aggregate stream is a **homogeneous Poisson process**.
* The aggregate arrival rate is the linear sum:

$$
\Lambda = \sum_{k=1}^K \lambda_k
$$

* The probability that the next arriving job originated from service $j$ is:

$$
P(\text{Origin} = j) = \frac{\lambda_j}{\Lambda}
$$

independent of the past history of arrivals.

```
Service 1 (lambda_1) -----\
Service 2 (lambda_2) ------> [ MERGE ] -----> Unified Stream: Lambda = lambda_1 + lambda_2 + lambda_3
Service 3 (lambda_3) -----/
```

### 2. Splitting (Thinning via Independent Bernoulli Routing)
Suppose an incoming Poisson stream with rate $\lambda$ is routed across $M$ downstream worker pools. Each arriving job is independently dispatched to pool $j$ with probability $p_j$, where $\sum_{j=1}^M p_j = 1$.

```
                                 /----> Pool 1: Poisson(lambda * p_1)
Unified Stream                  /
Poisson(lambda) ---> [ ROUTER ] ------> Pool 2: Poisson(lambda * p_2)
                                \
                                 \----> Pool 3: Poisson(lambda * p_3)
```

The mathematical result:
* Each individual output stream $j$ is an **exact Poisson process** with rate:

$$
\lambda_j = \lambda p_j
$$

* Crucially, the $M$ thinned output streams are **completely mutually independent**! Even though they originated from the same stream, observing 100 arrivals at Pool 1 gives zero information about how many arrivals reached Pool 2.

---

## 5. PASTA: Poisson Arrivals See Time Averages

One of the most celebrated and practically critical theorems in queueing theory is **PASTA (Poisson Arrivals See Time Averages)**, formalized by Ronald W. Wolff in 1982.

### The Theorem
In any queueing system where arrivals follow a Poisson process and future arrival increments are independent of the current state of the system:

> **The state distribution of the system observed by an arriving job at the instant of its arrival is identically equal to the long-term time-average state distribution of the system.**

Mathematically, let $P_n$ be the proportion of time the system has $n$ jobs in the long run:

$$
P_n = \lim_{T \to \infty} \frac{1}{T} \int_0^T \mathbf{1}_{\{N(t) = n\}} \, dt
$$

and let $\alpha_n$ be the proportion of arriving customers who find $n$ jobs in the system upon arrival:

$$
\alpha_n = \lim_{K \to \infty} \frac{1}{K} \sum_{k=1}^K \mathbf{1}_{\{N(a_k^-) = n\}}
$$

where $a_k^-$ is the instant immediately prior to the $k$-th arrival.

Wolff's theorem states:

$$
\alpha_n = P_n \quad \text{for all } n \ge 0
$$

### Why Non-Poisson Arrivals Violate PASTA
To appreciate PASTA, examine how deterministic or synchronized non-Poisson arrivals completely violate it:

#### Counter-Example 1: The Synchronized Cron Job
Consider a single build server with service duration $\mathbb{E}[S] = 1$ minute.
* Every hour on the hour ($t = 0, 60, 120, \dots$), an automated cron script triggers 30 builds simultaneously.
* For the first 30 minutes of every hour, the server is busy processing the backlog.
* For the remaining 30 minutes of every hour, the server is completely idle.
* **Time-Average State**: The server is busy 50% of the time ($P_1 = 0.50$, $P_0 = 0.50$).
* **Arrival-Average State**: Every arriving job arrives during the hourly burst. The first job sees 0 jobs, the second sees 1, the third sees 2, ..., the 30th sees 29! The average queue depth seen by an arriving job is $\frac{0 + 1 + \dots + 29}{30} = 14.5$ jobs.
* Here, $\alpha \neq P$. The arrivals see massive congestion because arrivals are correlated with each other.

#### Counter-Example 2: The Paced Deterministic Stream ($D/D/1$)
Consider a server where tests take exactly 4 minutes. A load balancer paces arrivals so a job arrives every 5 minutes.
* **Time-Average State**: The server is busy 4 out of every 5 minutes. Utilization is $P_1 = 0.80$ ($80\%$).
* **Arrival-Average State**: Every job arrives at $t = 5, 10, 15, \dots$, exactly 1 minute *after* the previous job finished! Every arriving job sees an empty server ($\alpha_0 = 1.0$, $\alpha_1 = 0.0$).
* The arriving jobs never wait in queue ($W_q = 0$), even though the server is busy 80% of the time.

Only a Poisson process—because its arrival epochs are uniformly distributed across time conditionally on the count—samples the state of the universe without bias.

---

## 6. Concrete Engineering Walkthrough: Distributed Webhook Ingestion

Let us apply Poisson mechanics to a real-world multi-service webhook ingestion gateway.

### The Scenario
A central CI/CD orchestrator receives webhook pull-request triggers from 4 independent repository microservices:
1. `Frontend Services`: $\lambda_1 = 3$ PRs/hour
2. `Backend APIs`: $\lambda_2 = 6$ PRs/hour
3. `Database Schemas`: $\lambda_3 = 2$ PRs/hour
4. `Infrastructure & Configs`: $\lambda_4 = 1$ PR/hour

### Step 1: Merging the Streams
Because the 4 services operate independently, the aggregate webhook stream entering the gateway is a homogeneous Poisson process with rate:

$$
\Lambda = \lambda_1 + \lambda_2 + \lambda_3 + \lambda_4 = 3 + 6 + 2 + 1 = \mathbf{12 \text{ PRs/hour}}
$$

In minutes: $\Lambda = \frac{12}{60} = \mathbf{0.20 \text{ PRs/minute}}$ (one PR every 5 minutes on average).

### Step 2: Probability of Silence vs Surge in a 15-Minute Window
Evaluate the probability of events in a 15-minute window ($t = 15$ min, expected arrivals $\mu_t = \Lambda t = 0.20 \times 15 = 3.0$ PRs):

1. **Probability of Complete Silence ($k = 0$)**:
   $$\mathbb{P}(N(15) = 0) = \frac{3^0 e^{-3}}{0!} = e^{-3} \approx \mathbf{0.0498 \quad (4.98\%)}$$
   Roughly 5% of 15-minute intervals will see zero PRs.

2. **Probability of Exactly Expected Traffic ($k = 3$)**:
   $$\mathbb{P}(N(15) = 3) = \frac{3^3 e^{-3}}{3!} = \frac{27 \times 0.049787}{6} \approx \mathbf{0.2240 \quad (22.4\%)}$$

3. **Probability of a Surge ($k \ge 6$ PRs, double the mean)**:
   $$\mathbb{P}(N(15) \ge 6) = 1 - \sum_{k=0}^5 \frac{3^k e^{-3}}{k!}$$
   Evaluating the terms:
   * $k=0: 0.0498$
   * $k=1: 3 \times 0.0498 = 0.1494$
   * $k=2: \frac{9}{2} \times 0.0498 = 0.2240$
   * $k=3: 0.2240$
   * $k=4: \frac{81}{24} \times 0.0498 = 0.1680$
   * $k=5: \frac{243}{120} \times 0.0498 = 0.1008$
   * Sum for $k=0 \dots 5 \approx 0.9160$
   $$\mathbb{P}(N(15) \ge 6) = 1 - 0.9160 = \mathbf{0.0840 \quad (8.40\%)}$$
   An engineer must expect an arrival surge of 6 or more PRs in an arbitrary 15-minute window roughly 8.4% of the time!

### Step 3: Thinning to a Hardware Testbed
Suppose that only 25% of pull requests require execution on physical hardware testbeds ($p_{\text{hw}} = 0.25$), while the other 75% run in virtualized containers.

Using the thinning property:
* The arrival stream of hardware test requests is a Poisson process with rate:

$$
\lambda_{\text{hw}} = \Lambda \cdot p_{\text{hw}} = 12 \times 0.25 = \mathbf{3.0 \text{ PRs/hour}}
$$

* The mean inter-arrival time between hardware test requests is:

$$
\mathbb{E}[X_{\text{hw}}] = \frac{1}{3.0} \text{ hours} = \mathbf{20.0 \text{ minutes}}
$$

---

## 7. Hands-On 3-Tier Scaffolding Challenges

### 🟢 Warmup (Tier 1): Cloud API Gateway Surge Bounding
**Scenario**: An API gateway receives requests at an average rate of $\lambda = 100$ requests per second. The gateway has a rate limiter buffer that can absorb up to $130$ requests in any 1-second interval without dropping packets.

1. What is the mean and standard deviation of the number of requests arriving in a 1-second window?
2. Using the Gaussian approximation to the Poisson distribution ($\mu = 100, \sigma = 10$), calculate the $z$-score corresponding to $130$ requests.
3. Compute the approximate probability that the rate limiter drops requests in any given 1-second interval.

<details>
<summary><b>Click here to view the solution...</b></summary>

**Solution**:
1. **Mean and Standard Deviation**:
   For a Poisson process with $\lambda = 100$ req/s over $t = 1.0$ second:
   $$\mu = \lambda t = 100.0$$
   $$\sigma = \sqrt{\lambda t} = \sqrt{100} = \mathbf{10.0 \text{ requests}}$$

2. **$Z$-Score**:
   With continuity correction (testing $N \ge 131$ or $130.5$):
   $$z = \frac{130.5 - 100}{10} = \frac{30.5}{10} = \mathbf{3.05}$$

3. **Drop Probability**:
   Using the standard normal complementary CDF $Q(z) = 1 - \Phi(z)$:
   $$\mathbb{P}(Z \ge 3.05) \approx \mathbf{0.00114 \quad (0.114\%)}$$
   A drop event occurs roughly once every 877 seconds (approx. once every 14.6 minutes).
</details>

---

### 🟡 Diagnostic (Tier 2): Quantifying PASTA Violation in Batch Merges
**Scenario**: A build farm has 1 server. Service times are deterministic: each build takes exactly $S = 5$ minutes.
* **Architecture A (Poisson Stream)**: Builds arrive as a Poisson process with mean rate $\lambda = 0.16$ builds/min ($\rho = 0.16 \times 5 = 0.80$).
* **Architecture B (Periodic Batch Cron)**: Exactly 8 builds arrive simultaneously every 50 minutes. (Mean arrival rate $\lambda = 8 / 50 = 0.16$ builds/min, utilization $\rho = 40 / 50 = 0.80$).

1. In Architecture A, what fraction of arriving builds find the server busy upon arrival?
2. In Architecture B, what fraction of arriving builds find the server busy upon arrival?
3. Calculate the average queue waiting time $W_q$ experienced by builds in Architecture B versus Architecture A ($M/D/1$ formula: $W_q = \frac{\rho S}{2(1 - \rho)}$).

<details>
<summary><b>Click here to view the solution...</b></summary>

**Solution**:
1. **Architecture A (Poisson Arrivals)**:
   By the PASTA theorem, arriving builds see time averages:
   $$\alpha_1 = P_1 = \rho = \mathbf{0.80 \quad (80\%)}$$
   Exactly $80\%$ of arriving builds find the server busy.

2. **Architecture B (Periodic Batch Arrivals)**:
   Every 50 minutes, 8 builds arrive simultaneously at an empty server:
   * Build 1 finds 0 jobs in the system (finds server IDLE).
   * Builds 2, 3, 4, 5, 6, 7, 8 find the server BUSY.
   Therefore, 7 out of 8 arriving builds find the server busy:
   $$\alpha_{\text{busy}} = \frac{7}{8} = \mathbf{0.875 \quad (87.5\%)}$$
   Arriving builds see a higher busy probability ($87.5\%$) than the time-average utilization ($80\%$)!

3. **Waiting Time Comparison**:
   * **In Architecture B**:
     * Build 1 waits: $0$ min
     * Build 2 waits: $5$ min
     * Build 3 waits: $10$ min
     * ...
     * Build 8 waits: $7 \times 5 = 35$ min
     Average wait:
     $$W_q(B) = \frac{0 + 5 + 10 + 15 + 20 + 25 + 30 + 35}{8} = \frac{140}{8} = \mathbf{17.5 \text{ minutes}}$$
   * **In Architecture A ($M/D/1$)**:
     $$W_q(A) = \frac{\rho S}{2(1 - \rho)} = \frac{0.80 \times 5}{2(1 - 0.80)} = \frac{4.0}{0.40} = \mathbf{10.0 \text{ minutes}}$$
   Batching the arrivals increases developer waiting time from 10.0 minutes to 17.5 minutes (a $75\%$ latency penalty)!
</details>

---

### 🔴 Capstone Challenge (Tier 3): Non-Homogeneous Poisson Arrival Rate Estimation
**Scenario**: In real-world software fleets, arrival rates vary across the day according to a non-homogeneous Poisson process (NHPP) with rate function $\lambda(t)$.
Suppose the arrival rate follows a diurnal sinusoidal profile:
$$\lambda(t) = \lambda_0 + A \sin\left(\frac{2\pi t}{24}\right)$$
with baseline $\lambda_0 = 20$ PRs/hour, amplitude $A = 12$ PRs/hour, and $t \in [0, 24]$ hours.

1. Find the peak arrival rate $\lambda_{\max}$, the minimum arrival rate $\lambda_{\min}$, and the times $t$ at which they occur.
2. Calculate the total expected number of PR arrivals $\Lambda_{\text{day}}$ over the 24-hour cycle.
3. Formulate Lewis and Shedler's **Thinning Algorithm** to generate exact sample paths of this non-homogeneous Poisson process using a homogeneous Poisson generator.

<details>
<summary><b>Click here to view the solution...</b></summary>

**Solution**:
1. **Extrema**:
   * Peak occurs when $\sin\left(\frac{2\pi t}{24}\right) = +1 \implies \frac{2\pi t}{24} = \frac{\pi}{2} \implies t = 6$ hours.
     $$\lambda_{\max} = 20 + 12 = \mathbf{32 \text{ PRs/hour}}$$
   * Minimum occurs when $\sin\left(\frac{2\pi t}{24}\right) = -1 \implies \frac{2\pi t}{24} = \frac{3\pi}{2} \implies t = 18$ hours.
     $$\lambda_{\min} = 20 - 12 = \mathbf{8 \text{ PRs/hour}}$$

2. **Total Expected Arrivals**:
   $$\Lambda_{\text{day}} = \int_0^{24} \left(20 + 12 \sin\left(\frac{2\pi t}{24}\right)\right) dt$$
   Because the integral of a complete sine period over $[0, 24]$ is 0:
   $$\Lambda_{\text{day}} = 20 \times 24 + 0 = \mathbf{480 \text{ PRs}}$$

3. **Lewis-Shedler Thinning Algorithm**:
   To generate events from NHPP with rate $\lambda(t) \le \lambda_{\max}$:
   1. Generate homogeneous candidate arrival times $t_1, t_2, \dots$ from a Poisson process with constant rate $\lambda^* = \lambda_{\max} = 32$ PRs/hour.
   2. For each candidate arrival $t_i \in [0, 24]$:
      * Compute acceptance probability $p(t_i) = \frac{\lambda(t_i)}{\lambda^*} = \frac{20 + 12 \sin(2\pi t_i / 24)}{32}$.
      * Draw an independent uniform random variable $U_i \sim \text{Uniform}(0, 1)$.
      * If $U_i \le p(t_i)$, accept $t_i$ as an arrival; otherwise, discard (thin) it.
   The resulting thinned points constitute an exact realization of the non-homogeneous Poisson process $\lambda(t)$.
</details>

---

**[Course Home](../README.md) | [Python Lab](../python/02_poisson_processes_and_memoryless_arrivals.ipynb) | [Prev: Ch 01: Conservation of Flow](01_little_law_and_conservation_of_flow.ipynb) | [Next: Ch 03: Birth-Death Chains](03_birth_death_chains_and_mm1_mminf.ipynb)**
